In [18]:
import transformers
import torch

In [19]:

model_id =  "meta-llama/Meta-Llama-3-8B-Instruct" # "meta-llama/Meta-Llama-3-8B" #

pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device="cpu",# cuda
)

Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.12it/s]
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [20]:
prompt = """
Convert the following medical trial eligibility criteria into a structured JSON format:
Inclusion Criteria:
- Overweight or obese subjects [according to body mass index (BMI)]
- Fasting plasma glucose value between 100 and 125 mg/dl, with impaired fasting glucose or impaired glucose tolerance confirmed with oral glucose tolerance test (OGTT)
- Total cholesterol values ≥ 200 mg/dl

Exclusion Criteria:
- Patients with neoplastic and liver diseases, renal failure
- Patients with type 1 or 2 diabetes mellitus
- Pregnant or breastfeeding women
- Hypersensitivity to any of the ingredients
- Therapy with lipid-lowering drugs
- Use of products containing red yeast rice
"""

template = "{\"InclusionCriteria\":{\"IC1\":\"...\",\"IC2\":\"...\",\"ICx\":\"...\"},\"ExclusionCriteria\":{\"EC1\":\"...\",\"EC2\":\"...\",\"EC3\":\"...\",\"ECx\":\"...\",}}"

messages = [
    {"role": "system", "content": f"You are a Elegibility Criteria to JSON machine. Return inclusion and exclusion criteria as JSON object. For every critera a key and the description as values. Take only the descriptions, add nothing extra, and number them. Use this template: {template}"},
    {"role": "user", "content": f"{prompt}"},
]

prompt = pipeline.tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
)

terminators = [
    pipeline.tokenizer.eos_token_id,
    pipeline.tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

In [21]:
outputs = pipeline(
    prompt,
    max_new_tokens=500,
    eos_token_id=terminators,
    do_sample=True,
    temperature=0.6,
    top_p=0.9,
)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [22]:
print(outputs[0]["generated_text"][len(prompt):])

Here is the eligibility criteria in a structured JSON format:

{
"Inclusion Criteria": [
  "1. Overweight or obese subjects [according to body mass index (BMI)]",
  "2. Fasting plasma glucose value between 100 and 125 mg/dl, with impaired fasting glucose or impaired glucose tolerance confirmed with oral glucose tolerance test (OGTT)",
  "3. Total cholesterol values ≥ 200 mg/dl"
],
"Exclusion Criteria": [
  "1. Patients with neoplastic and liver diseases, renal failure",
  "2. Patients with type 1 or 2 diabetes mellitus",
  "3. Pregnant or breastfeeding women",
  "4. Hypersensitivity to any of the ingredients",
  "5. Therapy with lipid-lowering drugs",
  "6. Use of products containing red yeast rice"
]
}
